# Bahrain Rental Price Prediction

Student-friendly, step-by-step version of the competition script.

**What it does**: trains a Random Forest to predict monthly rental prices (BHD)
and writes a submission file (`submission_v6.csv`).

**Key ideas**:
- **MedianForest**: predicts the *median* of all tree predictions instead of the
  mean \— optimal for Mean Absolute Error (MAE).
- **TargetEncoder** (`cv=5`) replaces the manual hierarchical encoding.
- Target is **capped at 1850 BHD** (outliers removed), trained on the raw target.
- `Agent_name` is used only as a *count* feature \— target-encoding it hurt the score.


In [31]:
import os
import warnings
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, TargetEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_absolute_error

warnings.filterwarnings('ignore')

# Set global seed for reproducibility
SEED = 42



# 1. Custom MAE-Optimized Model

class MedianForest(RandomForestRegressor):
    """
    Standard Random Forest predicts the mean of individual trees.
    For MAE loss, taking the median across tree predictions is optimal.
    """
    def predict(self, X):
        X = np.asarray(X, dtype=np.float32, order='C')
        tree_preds = np.empty((len(self.estimators_), X.shape[0]))

        for i, tree in enumerate(self.estimators_):
            tree_preds[i] = tree.predict(X, check_input=False)

        return np.median(tree_preds, axis=0)


In [32]:

# 2. Load Datasets

def load_data():
    train_path = 'data.csv' if os.path.exists('data.csv') else 'Competition/data.csv'
    test_path = 'test.csv' if os.path.exists('test.csv') else 'Competition/test.csv'

    train_df = pd.read_csv(train_path).dropna(subset=['rent']).reset_index(drop=True)
    test_df = pd.read_csv(test_path)

    return train_df, test_df


raw_tr, raw_te = load_data()
print(f"Data Loaded: Train {raw_tr.shape}, Test {raw_te.shape}")


Data Loaded: Train (10575, 16), Test (3527, 15)


In [33]:

# 3. Feature Engineering
# Text keywords to search for 
TITLE_PATTERNS = {
    'sea_view': r'sea\s*view',
    'new': r'\bnew\b',
    'brand_new': r'brand\s*new',
    'renov': r'reno(?:vat)',
    'compound': r'compound',
    'luxury': r'luxur',
    'spacious': r'spacious',
    'furn': r'furnish',
    'semi_furn': r'semi[\s-]*furnish',
    'inclusive': r'inclusive',
    'ewa': r'\bewa\b|electricity',
    'pool': r'\bpool\b',
    'gym': r'\bgym\b',
    'penthouse': r'penthouse',
    'duplex': r'duplex',
    'water': r'beach|waterfront|marina|lagoon',
    'view': r'\bview\b',
    'family': r'family',
    'exclusive': r'exclusive',
    'invest': r'investment|roi|yield',
    'sale': r'for\s*sale|forsale|\bsell\b|freehold|title\s*deed'
}

CATEGORICAL_COLS = ['Area', 'Governorate', 'Property_type', 'Agency', 'Agent_name', 'Include_w_e']


def featurize(df):
    d = df.copy()

    # text columns
    title_text = d['Title'].astype('string').fillna('').str.lower()
    beds_text = d['Beds'].astype('string').fillna('').str.lower()
    baths_text = d['Baths'].astype('string').fillna('').str.lower()
    size_text = d['Size'].astype('string').fillna('').str.lower()
    url_text = d['URL'].astype('string').fillna('').str.lower()
    offer_text = d['Offer'].astype('string').fillna('').str.lower()

    # Extract numerical counts from text
    num_regex = r'(\d+\.?\d*)'
    d['beds'] = beds_text.str.extract(num_regex, expand=False).astype('float64')
    d['beds'] = d['beds'].mask(beds_text.str.contains('studio'), 0.5)
    d['maid'] = beds_text.str.contains('maid').astype('int8')

    d['baths'] = baths_text.str.extract(num_regex, expand=False).astype('float64')
    d['baths'] = d['baths'].mask(baths_text == 'none', 0.0)

    d['sqft'] = size_text.str.extract(r'([\d,]+)\s*sqft', expand=False).str.replace(',', '').astype('float64')
    d['sqm'] = size_text.str.extract(r'([\d,]+)\s*sqm', expand=False).str.replace(',', '').astype('float64')
    d['unit_ratio'] = d['sqft'] / d['sqm'].replace(0, np.nan)

    # Dates & Availability
    dt = pd.to_datetime(d['Availability_date'], dayfirst=True, errors='coerce')
    d['avail_days'] = dt.astype('int64').where(dt.notna()) / 8.64e13
    d['avail_month'] = dt.dt.month
    d['avail_dow'] = dt.dt.dayofweek
    d['avail_missing'] = dt.isna().astype('int8')

    # Keyword features from title
    for name, pattern in TITLE_PATTERNS.items():
        d[f't_{name}'] = title_text.str.contains(pattern, na=False).astype('int8')

    d['t_len'] = title_text.str.len()
    d['t_words'] = title_text.str.count(r'\s+') + 1
    d['t_bars'] = title_text.str.count(r'\|')

    # URL & Offer features
    d['url_depth'] = url_text.str.count('/')
    d['url_sale'] = url_text.str.contains(r'/buy/|/sale/|for-sale').astype('int8')
    d['url_type'] = url_text.str.extract(r'/plp/[a-z]+/([a-z-]+)/', expand=False).fillna('NA')
    d['offer_sale'] = offer_text.str.contains('sale|buy').astype('int8')

    # Room & Ratio Combinations
    d['rooms'] = d['beds'].fillna(0) + d['baths'].fillna(0)
    d['sqft_per_bed'] = d['sqft'] / (d['beds'] + 1)
    d['sqft_per_room'] = d['sqft'] / (d['rooms'] + 1)
    d['baths_per_bed'] = d['baths'] / (d['beds'] + 1)
    d['log_sqft'] = np.log1p(d['sqft'])

    # Amenities Count
    amen_numeric = pd.to_numeric(d['Amenities'], errors='coerce')
    if amen_numeric.notna().mean() > 0.5:
        d['amen'] = amen_numeric
    else:
        d['amen'] = d['Amenities'].astype('string').fillna('').str.count(',') + 1

    # Categorical combinations
    for col in CATEGORICAL_COLS:
        d[col] = d[col].astype('string').fillna('NA')

    d['area_type'] = d['Area'] + '|' + d['Property_type']
    d['area_beds'] = d['Area'] + '|' + d['beds'].fillna(-1).astype(str)

    return d


def create_shared_stats(train_df, test_df):
    """Computes joint summary statistics across train and test datasets."""
    both = pd.concat([train_df, test_df], keys=['tr', 'te'])

    group_cols = ['Area', 'Governorate', 'Property_type', 'Agency', 'Agent_name', 'area_type']
    for col in group_cols:
        both[f'{col}_cnt'] = both.groupby(col)[col].transform('size')

    med_cols = ['Area', 'Governorate', 'Property_type']
    for col in med_cols:
        both[f'{col}_med_sqft'] = both.groupby(col)['sqft'].transform('median')

    both['rel_sqft'] = both['sqft'] / both['Area_med_sqft']
    both['rel_sqft_gov'] = both['sqft'] / both['Governorate_med_sqft']
    both['sqft_pct_area'] = both.groupby('Area')['sqft'].rank(pct=True)
    both['beds_pct_area'] = both.groupby('Area')['beds'].rank(pct=True)
    both['gov_type_cnt'] = both.groupby(['Governorate', 'Property_type'])['sqft'].transform('size')

    return both.loc['tr'].copy(), both.loc['te'].copy()


In [34]:
# Apply feature engineering to both datasets
tr_feat = featurize(raw_tr)
te_feat = featurize(raw_te)
tr, te = create_shared_stats(tr_feat, te_feat)

y = tr['rent'].to_numpy()

# Define numeric columns
drop_cols = {
    'Property_id', 'rent', 'Title', 'URL', 'Size', 'Beds', 'Baths', 'Availability_date',
    'Amenities', 'Offer', 'sqm', 'Area', 'Governorate', 'Property_type', 'Agency',
    'Agent_name', 'Include_w_e', 'area_type', 'area_beds', 'url_type'
}
NUM_COLS = [c for c in tr.columns if c not in drop_cols and tr[c].dtype.kind in 'ifbu']
print(f"Using {len(NUM_COLS)} numeric features")


Using 56 numeric features


In [35]:

# 4. Pipeline & Model Definition
def build_pipeline():
    te_cols = ['Area', 'area_type', 'area_beds', 'Agency', 'Governorate']
    ohe_cols = ['Property_type', 'Include_w_e', 'Governorate', 'url_type']

    preprocessor = ColumnTransformer(
        transformers=[
            ('te', TargetEncoder(target_type='continuous', smooth='auto', cv=5, random_state=SEED), te_cols),
            ('ohe', OneHotEncoder(handle_unknown='infrequent_if_exist', min_frequency=0.004, sparse_output=False), ohe_cols),
            ('num', 'passthrough', NUM_COLS)
        ],
        verbose_feature_names_out=False
    )

    model = MedianForest(
        n_estimators=800,
        max_features=0.3,
        min_samples_leaf=1,
        random_state=SEED,
        n_jobs=-1
    )

    return Pipeline([('pre', preprocessor), ('rf', model)])


In [36]:

# 5. Main Execution
CAP = 1850
cap_mask = tr['rent'] <= CAP
tr_capped = tr[cap_mask].reset_index(drop=True)
y_capped = y[cap_mask]
print(f"Capped target at {CAP}: keeping {len(tr_capped)} / {len(tr)} rows")

# Cross-Validation
strat_labels = pd.qcut(np.log1p(y_capped), 10, labels=False, duplicates='drop')
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

CV_preds = np.empty(len(y_capped))
for train_idx, val_idx in cv.split(tr_capped, strat_labels):
    fold_model = build_pipeline()
    fold_model.fit(tr_capped.iloc[train_idx], y_capped[train_idx])
    CV_preds[val_idx] = fold_model.predict(tr_capped.iloc[val_idx])

CV_mae = mean_absolute_error(y_capped, CV_preds)
print(f" MAE (mf=0.3, msl=1, n=800): {CV_mae:.2f} BHD")

# Final Model Fit & Test Predictions
final_model = build_pipeline()
final_model.fit(tr_capped, y_capped)

test_preds = np.clip(final_model.predict(te), 0, None)
print(f"Predictions: min={test_preds.min():.0f}, median={np.median(test_preds):.0f}, max={test_preds.max():.0f}")

# Export CSV
submission = pd.DataFrame({
    'Property_id': raw_te['Property_id'],
    'rent': test_preds
})
submission.to_csv('submission_v6.csv', index=False)
print("Saved submission to submission_v6.csv")


Capped target at 1850: keeping 10448 / 10575 rows
 MAE (mf=0.3, msl=1, n=800): 80.69 BHD
Predictions: min=120, median=440, max=1800
Saved submission to submission_v6.csv
